# Study 2 Analyses

Code to generate the figures for Study 2.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['ytick.left'] = True
sns.set_style("ticks")

import helpers
import generalization_utils
import qualitative_utils
import regression_utils

## Ghost Voxels Generalization Heatmap

In [3]:
ghost_results = os.path.join("..", "results", "categories", "full_features", "generalization_8000", "sae", "gemma-2-2b-res-matryoshka-dc", "mean", "12", "standardized_betas", "generalization", "ghost.csv")
df = pd.read_csv(ghost_results)
similarity, voxels = generalization_utils.run_cluster_generalization_analysis(df)

In [4]:
fig = generalization_utils.plot_similarity_matrix(similarity, rectangle_edges=[(70, 102)], dpi=500)
fig.savefig("figures/preprint/study2_ghost_generalization.pdf", bbox_inches="tight")

In [5]:
shared_voxels = qualitative_utils.parse_voxel_list(voxels[70:102])

### Ghost Cluster Voxel Metadata

In [6]:
META_DIR = os.path.join("..", "data", "processed_csvs_anon")
DISPLAY_COLS = ["participant", "neuroid_id", "parc_name_glasser", "parc_lang",
                "tstat_langloc_SN", "ncsnr", "quadrant16",
                "corr_SentPC1", "corr_SentPC2", "visfAtlas_name"]

rows = []
for participant, neuroid in shared_voxels:
    path = os.path.join(META_DIR, participant, "ghost_meta.csv")
    df_meta = pd.read_csv(path)
    match = df_meta[df_meta.iloc[:, 0] == neuroid]
    if len(match) == 0:
        print(f"WARNING: {participant}_{neuroid} not found")
        continue
    row = match.iloc[0].to_dict()
    row["participant"] = participant
    rows.append(row)

ghost_cluster_meta = pd.DataFrame(rows)
# Move participant and neuroid_id to front
front = ["participant", "neuroid_id"]
ghost_cluster_meta = ghost_cluster_meta[front + [c for c in ghost_cluster_meta.columns if c not in front]]

print(f"Found metadata for {len(ghost_cluster_meta)} / {len(shared_voxels)} ghost cluster voxels\n")
ghost_cluster_meta[DISPLAY_COLS].round(3)

Found metadata for 32 / 32 ghost cluster voxels



,participant,neuroid_id,parc_name_glasser,parc_lang,tstat_langloc_SN,ncsnr,quadrant16,corr_SentPC1,corr_SentPC2,visfAtlas_name
0,cvn7016,86409,TPOJ2,5,6.692,0.409,5.0,0.011,0.029,MTG-bodies
1,cvn7016,86408,TPOJ2,5,7.398,0.409,5.0,0.064,-0.004,MTG-bodies
2,cvn7012,94279,PGi,6,2.455,0.433,10.0,-0.005,-0.012,NaN
3,cvn7012,130091,PGi,6,0.971,0.565,9.0,-0.006,0.018,NaN
4,cvn7009,48708,9m,0,3.710,0.445,5.0,-0.005,0.010,NaN
5,cvn7016,8202,9m,0,1.683,0.426,7.0,-0.008,-0.015,NaN
6,cvn7016,82883,SFL,0,0.985,0.405,7.0,-0.002,-0.011,NaN
7,cvn7016,77028,9m,0,2.551,0.413,6.0,0.021,0.020,NaN
8,cvn7016,17996,9m,0,3.210,0.429,5.0,0.015,0.045,NaN
9,cvn7016,77112,9m,0,1.887,0.411,6.0,-0.008,0.016,NaN


In [9]:
# Participant breakdown
counts = ghost_cluster_meta["participant"].value_counts()
pcts = (counts / len(ghost_cluster_meta) * 100).round(1)
breakdown = pd.DataFrame({"n_voxels": counts, "percent": pcts})
breakdown.index.name = "participant"
print(f"Ghost cluster: {len(ghost_cluster_meta)} voxels from {len(counts)} participants\n")
breakdown

Ghost cluster: 32 voxels from 6 participants



,n_voxels,percent
participant,,
cvn7012,12,37.5
cvn7016,9,28.1
cvn7009,6,18.8
cvn7007,3,9.4
cvn7002,1,3.1
cvn7011,1,3.1


In [10]:
# lang_parc breakdown
counts_parc_lang = ghost_cluster_meta["parc_lang"].value_counts()
pcts_parc_lang = (counts_parc_lang / len(ghost_cluster_meta) * 100).round(1)
breakdownparc_lang = pd.DataFrame({"n_voxels": counts_parc_lang, "percent": pcts_parc_lang})
breakdownparc_lang.index.name = "parc_lang"
print(f"Ghost cluster: {len(ghost_cluster_meta)} voxels from {len(counts_parc_lang)} parcels\n")
breakdownparc_lang


Ghost cluster: 32 voxels from 4 parcels



,n_voxels,percent
parc_lang,,
6,14,43.8
0,9,28.1
4,5,15.6
5,4,12.5


### Voxels Outside the Ghost Cluster (Comparison)

Same participant + parc_lang summaries as above, but for the voxels in the generalization ordering that fall *outside* the ghost cluster — i.e., the complement of `shared_voxels`. Useful as a baseline for the cluster's compositional skew.


In [7]:
# Voxels OUTSIDE the ghost cluster (rest of the generalization ordering)
all_voxels = qualitative_utils.parse_voxel_list(voxels)
shared_set = set(shared_voxels)
outside_voxels = [v for v in all_voxels if v not in shared_set]

rows = []
for participant, neuroid in outside_voxels:
    path = os.path.join(META_DIR, participant, "ghost_meta.csv")
    df_meta = pd.read_csv(path)
    match = df_meta[df_meta.iloc[:, 0] == neuroid]
    if len(match) == 0:
        print(f"WARNING: {participant}_{neuroid} not found")
        continue
    row = match.iloc[0].to_dict()
    row["participant"] = participant
    rows.append(row)

ghost_outside_meta = pd.DataFrame(rows)
front = ["participant", "neuroid_id"]
ghost_outside_meta = ghost_outside_meta[front + [c for c in ghost_outside_meta.columns if c not in front]]
print(f"Outside cluster: {len(ghost_outside_meta)} / {len(outside_voxels)} voxels found\n")

# parc_lang breakdown (outside)
counts_pl_out = ghost_outside_meta["parc_lang"].value_counts()
pcts_pl_out = (counts_pl_out / len(ghost_outside_meta) * 100).round(1)
breakdown_pl_out = pd.DataFrame({"n_voxels": counts_pl_out, "percent": pcts_pl_out})
breakdown_pl_out.index.name = "parc_lang"
print(f"By parc_lang ({len(counts_pl_out)} parcels):")
breakdown_pl_out

Outside cluster: 128 / 128 voxels found

By parc_lang (6 parcels):


,n_voxels,percent
parc_lang,,
0,111,86.7
5,9,7.0
4,4,3.1
6,2,1.6
1,1,0.8
2,1,0.8


In [8]:
# Glasser parcel breakdown (outside)
counts_g_out = ghost_outside_meta["parc_name_glasser"].value_counts(dropna=False)
pcts_g_out = (counts_g_out / len(ghost_outside_meta) * 100).round(1)
breakdown_g_out = pd.DataFrame({"n_voxels": counts_g_out, "percent": pcts_g_out})
breakdown_g_out.index.name = "parc_name_glasser"
print(f"By Glasser parcel ({len(counts_g_out)} parcels):")
breakdown_g_out


By Glasser parcel (64 parcels):


,n_voxels,percent
parc_name_glasser,,
9m,7,5.5
7PL,7,5.5
PFm,6,4.7
Unknown,5,3.9
MIP,5,3.9
...,...,...
IFSa,1,0.8
10v,1,0.8
LBelt,1,0.8


In [9]:
# Full meta (all columns) for the outside-cluster voxels
with pd.option_context("display.max_columns", None, "display.width", None):
    display(ghost_outside_meta.round(3))


,participant,neuroid_id,neuroid_id.1,uid,betas_mean,parc_lang,parc_glasser,parc_name_glasser,tstat_langloc_SN,betas_langloc_S,betas_langloc_N,tstat_speechloc_SN,tstat_speechloc_NQ,top10_tstat_langloc_SN_parc_lang_int,top10_tstat_speechloc_SN_parc_lang_int,ncsnr,nc,2PC_model_intercept_avg,2PC_model_coef_0_avg,2PC_model_coef_1_avg,R2_train_PC2_avg,R2_test_PC2_avg,tstat_langloc_speechloc_SN_mean,angles,quadrant16,visfAtlas,visfAtlas_name,parc_lang_btla,parc_lang_lVTC_antlVTC,neuroid_id.2,corr_SentPC1,corr_SentPC2,combined_abs_corr
0,cvn7011,153159,153159,cvn7011,-0.175,0,69,9m,0.100,-0.128,-0.160,-0.712,-0.516,0,0,0.415,34.030,-0.175,0.019,0.007,0.481,-5.065,-1.206,0.337,0.0,NaN,NaN,0.0,0,153159,-0.003,0.048,0.050
1,cvn7006,41755,41755,cvn7006,0.094,0,120,H,1.965,0.867,0.137,0.224,-1.673,0,0,0.417,34.317,0.094,0.031,-0.042,0.284,-3.531,-0.255,5.352,13.0,NaN,NaN,0.0,0,41755,0.025,-0.022,0.047
2,cvn7002,162350,162350,cvn7002,-0.220,0,135,TF,0.441,0.589,0.129,3.091,-1.217,0,0,0.424,35.050,-0.220,-0.086,-0.057,0.255,-11.925,-1.872,3.732,9.0,1.0,mFus-faces,0.0,1,162350,-0.028,-0.018,0.047
3,cvn7006,16939,16939,cvn7006,2.654,0,104,RI,1.454,0.252,-0.093,0.425,0.216,0,0,0.415,34.042,2.654,-0.005,0.031,0.346,-3.150,0.760,1.722,4.0,NaN,NaN,0.0,0,16939,0.012,0.046,0.058
4,cvn7006,104211,104211,cvn7006,0.561,0,104,RI,-0.249,0.507,0.549,1.790,-1.062,0,0,0.410,33.499,0.561,-0.013,0.043,0.789,-3.784,0.648,1.865,4.0,NaN,NaN,0.0,0,104211,0.000,0.074,0.074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,cvn7011,75517,75517,cvn7011,-0.413,0,134,TE2a,0.272,0.202,0.000,0.254,0.674,0,0,0.407,33.209,-0.413,0.030,0.073,0.389,-7.515,0.123,1.175,2.0,NaN,NaN,0.0,0,75517,0.014,0.039,0.053
124,cvn7002,63641,63641,cvn7002,0.078,0,46,7PL,0.830,-0.561,-0.703,-1.846,0.979,0,0,0.405,32.948,0.078,0.012,-0.005,0.447,-4.379,0.424,5.904,15.0,NaN,NaN,0.0,0,63641,0.023,-0.018,0.041
125,cvn7012,14089,14089,cvn7012,-0.058,0,9,3b,-0.417,0.014,0.069,-1.500,2.945,0,0,0.403,32.800,-0.058,0.006,0.014,0.572,-6.942,-0.545,1.190,3.0,NaN,NaN,0.0,0,14089,0.005,0.020,0.025
126,cvn7012,102980,102980,cvn7012,-0.031,0,1,V1,-0.564,-1.042,-0.891,-0.418,-0.009,0,0,0.405,32.982,-0.031,0.006,-0.018,0.557,-4.041,-0.284,5.038,12.0,NaN,NaN,0.0,0,102980,0.010,-0.023,0.032


## Ghost Voxels Qualitative Features

In [ ]:
path_to_results = os.path.join("..", "results", "categories", "full_features", "regressions", "sae", "gemma-2-2b-res-matryoshka-dc", "mean", "12", "standardized_betas", "regressions", "qualitative", "Qualitative_Analysis_8000_n20_per_participant.csv")
raw_feature_sharing_stats = qualitative_utils.analyze_raw_feature_sharing(path_to_results, datasets=["ghost"], voxels=shared_voxels)

In [ ]:
fig, ax = qualitative_utils.plot_top_features_heatmap(raw_feature_sharing_stats, max_voxels=len(shared_voxels))
# fig.savefig("figures/preprint/study2_ghost_heatmap.pdf", bbox_inches="tight")
# fig.savefig("figures/preprint/study2_ghost_heatmap.svg", bbox_inches="tight")

In [ ]:
fig, ax = qualitative_utils.plot_top_features_heatmap(raw_feature_sharing_stats, max_voxels=len(shared_voxels), format="landscape")
fig.savefig("figures/preprint/study2_ghost_heatmap_landscape.pdf", bbox_inches="tight")
fig.savefig("figures/preprint/study2_ghost_heatmap_landscape.svg", bbox_inches="tight")

## Appendix: Barplot for Ghost Voxels

In [ ]:
fig, ax = regression_utils.predictivity_bar_plot(filter_set=["Ghost"], figsize=(5, 4.2))
fig.savefig("figures/preprint/study2_appendix_ghost_barplot.pdf", bbox_inches="tight")